In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F

In [ ]:
# Load train/val/test datasets and create DataLoaders
import os
from pathlib import Path
import torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# --- Configuration ---
data_dir = Path("datasets/waste_class_split")
batch_size = 32
# safe default: 224x224 input size
input_size = 224

# Use ImageNet normalization if you plan to use pretrained/backbone; otherwise compute dataset mean/std
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(input_size, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize(int(input_size * 1.14)), 
    transforms.CenterCrop(input_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

# Quick checks
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory {data_dir} not found. Run the split cell first.")

for split in ["train", "val", "test"]:
    p = data_dir / split
    if not p.exists():
        raise FileNotFoundError(f"Expected split folder not found: {p}")

# Create ImageFolder datasets
train_ds = ImageFolder(data_dir / "train", transform=train_transform)
val_ds = ImageFolder(data_dir / "val", transform=val_transform)
test_ds = ImageFolder(data_dir / "test", transform=val_transform)

print(f"Found classes: {train_ds.classes}")
print(f"Train size: {len(train_ds)}, Val size: {len(val_ds)}, Test size: {len(test_ds)}")

# DataLoaders
num_workers = min(4, os.cpu_count() or 0)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

# Sanity check: get one batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)  # [B, C, H, W]
print("Batch labels shape:", labels.shape)
print("Device available:", device)

# Expose variables for downstream cells: train_loader, val_loader, test_loader, train_ds
